In [1]:
import bz2
import orjson
import langcodes

from itertools import islice
from pathlib import Path
from typing import Any, Iterable
from functools import lru_cache
from typing import TypedDict

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "wikidata"
WIKIDATA_PATH = RAW_DIR / "latest-all.json.bz2"

assert WIKIDATA_PATH.exists()

SAMPLE_LIMIT = 50_000

In [2]:

def json_loads_line(line: str) -> dict[str, Any]:
    return orjson.loads(line)

def iter_wikidata_entities(path: Path, limit: int | None = None) -> Iterable[dict[str, Any]]:
    """Stream entity objects from the Wikidata JSON dump."""
    seen = 0
    with bz2.open(path, "rt", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line or line in {"[", "]"}:
                continue
            if line.endswith(","):
                line = line[:-1]
            if not line:
                continue
            entity = json_loads_line(line)
            seen += 1
            yield entity
            if limit is not None and seen >= limit:
                break

def sample_entities(path: Path, limit: int = SAMPLE_LIMIT):
    return list(islice(iter_wikidata_entities(path), limit))

In [3]:
entities = sample_entities(WIKIDATA_PATH)

In [4]:
LOCATION_CLASS_QIDS: dict[str, str] = {
    # ------------------------------------------------------------------
    # Political / country-level entities
    # ------------------------------------------------------------------

    "Q6256": "country",              # country
    "Q3624078": "country",           # sovereign state
    "Q3024240": "historical_country",# historical country / former state

    # "Q7275": "state",              # state; broad political concept
    # "Q15634554": "limited_recognition_state"

    # ------------------------------------------------------------------
    # Administrative territorial entities
    # ------------------------------------------------------------------

    "Q56061": "admin_division",      # administrative territorial entity
    "Q108640": "admin_division",     # administrative territorial entity of a single country

    "Q34876": "province",            # province
    "Q28575": "county",              # county
    "Q149621": "district",           # district
    "Q35657": "state_subdivision",   # state of the United States
    "Q107390": "federated_state",    # federated state

    # "Q484170": "commune",          # commune
    # "Q2983893": "department",      # department
    # "Q15060255": "municipality_type", # too broad unless audited

    # ------------------------------------------------------------------
    # Human settlements
    # ------------------------------------------------------------------

    "Q486972": "settlement",         # human settlement
    "Q515": "city",                  # city
    "Q1549591": "city",              # big city
    "Q200250": "city",               # metropolis
    "Q174844": "city",               # megacity

    "Q3957": "town",                 # town
    "Q532": "village",               # village
    "Q5084": "hamlet",               # hamlet
    "Q15284": "municipality",        # municipality

    # ------------------------------------------------------------------
    # Water features
    # ------------------------------------------------------------------

    "Q4022": "river",                # river
    "Q23397": "lake",                # lake
    "Q165": "sea",                   # sea
    "Q9430": "ocean",                # ocean
    "Q39594": "bay",                 # bay
    "Q1322134": "gulf",              # gulf
    "Q37901": "strait",              # strait

    # ------------------------------------------------------------------
    # Land / relief features
    # ------------------------------------------------------------------

    "Q46831": "mountain_range",      # mountain range
    "Q93352": "coast",               # coast
    "Q34763": "peninsula",           # peninsula
    "Q23442": "island",              # island
    "Q33837": "archipelago",         # archipelago
    "Q8514": "desert",              # desert
    "Q8072": "volcano",             # volcano
}

LOCATION_KIND_PRIORITY = [
    "country",
    "historical_country",

    "city",
    "town",
    "municipality",
    "village",
    "hamlet",
    "settlement",

    "island",
    "archipelago",
    "peninsula",
    "coast",
    "mountain_range",
    "desert",
    "volcano",

    "river",
    "lake",
    "sea",
    "ocean",
    "bay",
    "gulf",
    "strait",

    "state_subdivision",
    "federated_state",
    "province",
    "county",
    "district",
    "admin_division",
]

In [5]:
def claim_values(entity: dict, prop: str) -> list[dict]:
    values = []

    for stmt in entity.get("claims", {}).get(prop, []):
        if stmt.get("rank") == "deprecated":
            continue

        mainsnak = stmt.get("mainsnak", {})
        if mainsnak.get("snaktype") != "value":
            continue

        datavalue = mainsnak.get("datavalue")
        if not datavalue:
            continue

        values.append({
            "value": datavalue.get("value"),
            "type": datavalue.get("type"),
            "datatype": mainsnak.get("datatype"),
            "rank": stmt.get("rank", "normal"),
        })

    return values


def item_ids(entity: dict, prop: str) -> list[str]:
    ids = []

    for v in claim_values(entity, prop):
        value = v["value"]
        if isinstance(value, dict) and value.get("id"):
            ids.append(value["id"])

    return ids

def classify_location(entity: dict) -> str | None:
    p31_qids = item_ids(entity, "P31")

    kinds = {
        LOCATION_CLASS_QIDS[qid]
        for qid in p31_qids
        if qid in LOCATION_CLASS_QIDS
    }

    if not kinds:
        return None

    for kind in LOCATION_KIND_PRIORITY:
        if kind in kinds:
            return kind

    return sorted(kinds)[0]

PREFERRED_LABEL_LANGS = ("en", "mul", "fr", "de", "es", "ru", "zh", "la")


def default_label(entity: dict, langs: tuple[str, ...] = PREFERRED_LABEL_LANGS) -> str | None:
    labels = entity.get("labels", {})

    for lang in langs:
        obj = labels.get(lang)
        if obj and obj.get("value"):
            return obj["value"]

    for obj in labels.values():
        if obj.get("value"):
            return obj["value"]

    return None

def string_values(entity: dict, prop: str) -> list[str]:
    values = []

    for v in claim_values(entity, prop):
        value = v["value"]
        if isinstance(value, str):
            values.append(value)

    return values

def coordinate_value(entity: dict, prop: str = "P625") -> tuple[float, float] | None:
    values = claim_values(entity, prop)
    if not values:
        return None

    preferred = [v for v in values if v.get("rank") == "preferred"]
    chosen = preferred[0] if preferred else values[0]

    value = chosen["value"]

    if not isinstance(value, dict):
        return None

    lat = value.get("latitude")
    lon = value.get("longitude")

    if lat is None or lon is None:
        return None

    return float(lat), float(lon)

def dedupe_preserve_order(values: list[str]) -> list[str]:
    seen = set()
    out = []

    for value in values:
        if value not in seen:
            seen.add(value)
            out.append(value)

    return out

locations: list[dict[str, Any]] = []
multi_language_data: list[dict[str, Any]] = []

def extract_label_rows(entity: dict[str, Any]) -> list[dict[str, Any]]:
    """
    Extract Wikidata labels for one accepted location entity.
    """
    qid = entity["id"]
    rows: list[dict[str, Any]] = []

    for lang, label_obj in entity.get("labels", {}).items():
        value = label_obj.get("value")
        if not value:
            continue

        rows.append({
            "qid": qid,
            "wd_lang": label_obj.get("language", lang),
            "name": value,
            "term_type": "label",
        })

    return rows

# Iterate over sampled Wikidata entities and keep only those that our P31-based
# classifier recognizes as location-like entities. P31 ("instance of") tells us
# what kind of thing the entity is: country, city, river, island, hamlet, etc.
#
# For each accepted entity, build a compact debug record containing only the
# location metadata we actually care about at this stage:
#   - qid: Wikidata item id, used as the stable entity identifier
#   - debug_name: a convenient human-readable label for inspection only
#   - kind: our coarse location category derived from accepted P31 values
#   - p31_qids: raw Wikidata class QIDs, preserved for auditing/reclassification
#   - geonames_ids: optional GeoNames bridge IDs from P1566
#   - lat/lon: optional coordinates from P625

for entity in entities:
    kind = classify_location(entity)
    if kind is None:
        continue

    coords = coordinate_value(entity)
    
    multi_language_data.extend(extract_label_rows(entity))

    location = {
        "qid": entity["id"],
        "debug_name": default_label(entity),
        "kind": kind,
        "p31_qids": dedupe_preserve_order(item_ids(entity, "P31")),
        "geonames_ids": dedupe_preserve_order(string_values(entity, "P1566")),
        "lat": coords[0] if coords else None,
        "lon": coords[1] if coords else None,
    }

    locations.append(location)

In [6]:
class LangNorm(TypedDict):
    wd_lang: str  # original WD code
    geo_lang: str | None  # Geonames compatible code
    iso639_3: str | None  # ISO 639-3 three-letter, or None if unresolvable
    iso639_1: str | None  # ISO 639-1 two-letter where one exists


WIKIDATA_LANG_OVERRIDES: dict[str, str] = {
    "als": "gsw",  # Alemannic German (NOT Tosk Albanian)
    "simple": "en",  # Simple English Wikipedia
    "be-x-old": "be",  # Belarusian (Taraškievica) — variant of be
    "be-tarask": "be",
    "zh-classical": "lzh",  # Literary Chinese
    "zh-min-nan": "nan",  # Min Nan
    "zh-yue": "yue",  # Cantonese
    "fiu-vro": "vro",  # Võro
    "bat-smg": "sgs",  # Samogitian
    "roa-rup": "rup",  # Aromanian
    "nrm": "nrf",  # Norman
    "cbk-zam": "cbk",  # Chavacano de Zamboanga
}

SPECIAL_GEO_LANG_OVERRIDES: dict[str, str] = {
    "map-bms": "map-bms",
    "roa-tara": "roa-tara"
}

HARDCODED_LANG_NORMS: dict[str, LangNorm] = {
    "tl": LangNorm(wd_lang="tl", geo_lang="tl", iso639_3="tgl", iso639_1="tl"),
    # Wikidata's `tl` means Tagalog. langcodes maps tl/tgl/fil -> fil
    # because BCP 47 deprecated tl in favor of fil, but ISO 639-3 keeps
    # them distinct (tgl=Tagalog, fil=Filipino as standardized variety).
}

@lru_cache(maxsize=8192)
def normalize_wd_lang(wd_code: str) -> LangNorm:
    code = wd_code.strip().lower()

    if code in HARDCODED_LANG_NORMS:
        return HARDCODED_LANG_NORMS[code]
    
    if code in SPECIAL_GEO_LANG_OVERRIDES:
        return LangNorm(
            wd_lang=code,
            geo_lang=SPECIAL_GEO_LANG_OVERRIDES[code],
            iso639_3=None,
            iso639_1=None,
        )

    if code in WIKIDATA_LANG_OVERRIDES:
        canonical_input = WIKIDATA_LANG_OVERRIDES[code]
    else:
        base, sep, rest = code.partition("-")
        if base in WIKIDATA_LANG_OVERRIDES:
            canonical_input = WIKIDATA_LANG_OVERRIDES[base] + (
                sep + rest if sep else ""
            )
        else:
            canonical_input = code

    try:
        lang = langcodes.Language.get(canonical_input)

        try:
            iso3 = lang.to_alpha3()
        except LookupError:
            iso3 = None

        primary = lang.language  # could be 2 or 3 letters
        iso1 = primary if primary and len(primary) == 2 else None

        geo_lang = iso1 or iso3

        return LangNorm(
            wd_lang=code,
            geo_lang=geo_lang,
            iso639_3=iso3,
            iso639_1=iso1
        )
    except Exception:
        return LangNorm(
            wd_lang=code,
            geo_lang=None,
            iso639_3=None,
            iso639_1=None
        )

# Build a normalization map for the Wikidata/Wikimedia language codes that
# appeared in the extracted multilingual label rows.
#
# Wikidata labels use `wd_lang` codes such as "en", "fr", "zh-hant",
# "be-tarask", "sr-el", etc. These are useful as source/provenance codes, but
# for NTE and GeoNames-style lookup we also want a simpler language bucket:
# usually ISO-639-1 when available, otherwise ISO-639-3, with a few explicit
# Wikimedia-specific exceptions.
#
# We first collect the distinct `wd_lang` values so each language code is
# normalized only once, then store the result in `lang_norms` for later use when
# writing the names table.

wd_langs: set[str] = set()

for row in multi_language_data:
    wd_langs.add(row["wd_lang"])

lang_norms: dict[str, LangNorm] = {}
for wd_lang in wd_langs:
    lang_norm = normalize_wd_lang(wd_lang)
    lang_norms[wd_lang] = lang_norm

In [7]:
for row in multi_language_data:
    qid = row["qid"]
    wd_lang = row["wd_lang"]
    obj = {
        "qid": qid,
        "wd_lang": wd_lang,
        "geo_lang": lang_norms[wd_lang]["geo_lang"],
        "name": row["name"]
    }
    # Output Belgium names for diagnostic
    if qid == "Q31":
        print(obj)

{'qid': 'Q31', 'wd_lang': 'el', 'geo_lang': 'el', 'name': 'Βέλγιο'}
{'qid': 'Q31', 'wd_lang': 'ay', 'geo_lang': 'ay', 'name': 'Bilkiya'}
{'qid': 'Q31', 'wd_lang': 'pnb', 'geo_lang': 'pnb', 'name': 'بیلجیئم'}
{'qid': 'Q31', 'wd_lang': 'na', 'geo_lang': 'na', 'name': 'Berdjiyum'}
{'qid': 'Q31', 'wd_lang': 'mk', 'geo_lang': 'mk', 'name': 'Белгија'}
{'qid': 'Q31', 'wd_lang': 'bn', 'geo_lang': 'bn', 'name': 'বেলজিয়াম'}
{'qid': 'Q31', 'wd_lang': 'bpy', 'geo_lang': 'bpy', 'name': 'বেলজিয়াম'}
{'qid': 'Q31', 'wd_lang': 'lt', 'geo_lang': 'lt', 'name': 'Belgija'}
{'qid': 'Q31', 'wd_lang': 'jam', 'geo_lang': 'jam', 'name': 'Beljiom'}
{'qid': 'Q31', 'wd_lang': 'sk', 'geo_lang': 'sk', 'name': 'Belgicko'}
{'qid': 'Q31', 'wd_lang': 'so', 'geo_lang': 'so', 'name': 'Beljim'}
{'qid': 'Q31', 'wd_lang': 'tl', 'geo_lang': 'tl', 'name': 'Belgium'}
{'qid': 'Q31', 'wd_lang': 'uk', 'geo_lang': 'uk', 'name': 'Бельгія'}
{'qid': 'Q31', 'wd_lang': 'tt', 'geo_lang': 'tt', 'name': 'Бельгия'}
{'qid': 'Q31', 'wd_lang